# Processamento de Dados dos Influenciadores

Este notebook realiza a extração, tratamento e estruturação dos dados brutos em camadas analíticas.

**Decisão Metodológica Central**: Os posts com `comments_count == 0` NÃO são filtrados da base principal, pois representam uma classe válida de engajamento na plataforma (90%+ dos dados). A filtragem de zeros ocorrerá apenas em uma camada específica (`core_2017_2019`) voltada para testes de robustez.

## Camadas de Dados Geradas
1. `posts_base_completa.parquet`: Tratamento mínimo, sem filtros.
2. `posts_base_2017_2019.parquet`: **DATASET ANALÍTICO PRINCIPAL**. Filtro de data (2017-2019) e followers > 0.
3. `posts_core_2017_2019.parquet`: Subconjunto do principal, com `comments_count > 0` para testes de robustez.
4. Camadas de CV (`cv_core_2017_2019.parquet`, `cv_core_anual.parquet`): Métricas de estabilidade.


In [26]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pl.Config.set_tbl_rows(10)
pl.Config.set_fmt_str_lengths(50)

# Caminhos de entrada
PARQUET_POSTS = "posts.parquet"
CSV_INFLUENCERS = "dataframe_influencers.csv"

# Caminhos de saída
PARQUET_BASE_COMPLETA = "posts_base_completa.parquet"
PARQUET_BASE_2017_2019 = "posts_base_2017_2019.parquet"
PARQUET_CORE_2017_2019 = "posts_core_2017_2019.parquet"
PARQUET_CV_CORE = "cv_core_2017_2019.parquet"
PARQUET_CV_ANUAL = "cv_core_anual.parquet"

# Ordenações
ORDEM_BUCKET = ['nano', 'micro', 'mid-tier', 'macro', 'mega']
ORDEM_ER_IND = ['LOW', 'MEDIUM', 'HIGH', 'VIRAL']
ORDEM_ER_PERC = ['Q1', 'Q2', 'Q3', 'Q4']

## 1. Carga da Base Bruta

In [27]:
df = pl.read_parquet(PARQUET_POSTS)
print(f"Shape: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print(f"Tamanho estimado: {df.estimated_size('mb'):.2f} MB")
df.head()

Shape: 9,726,198 linhas x 20 colunas
Tamanho estimado: 1393.76 MB


post_id,username,inf_category,followers,followees,posts_total,is_verified,data,timestamp_unix,likes,comments_count,post_type,n_imagens,aspect_ratio,is_sponsored,n_hashtags,n_usertags,caption_len,has_location,accessibility
str,str,str,i64,i64,i64,bool,str,i64,i64,i64,str,i64,f64,bool,i64,i64,i64,bool,str
"""BmlIb5Xld5f""","""00s_supermodels""","""fashion""",2031,53,2520,false,"""2018-08-17""",1534509290,51,2,"""image""",1,0.8,false,18,1,257,false,""""""
"""BmlI2LPlblP""","""00s_supermodels""","""fashion""",2031,53,2520,false,"""2018-08-17""",1534509505,89,0,"""image""",1,0.8,false,17,1,279,false,""""""
"""BmoH1VclTxO""","""00s_supermodels""","""fashion""",2031,53,2520,false,"""2018-08-18""",1534609638,94,0,"""image""",1,0.8,false,16,1,228,false,""""""
"""Bmd0fuolkUZ""","""00s_supermodels""","""fashion""",2031,53,2520,false,"""2018-08-14""",1534263955,66,4,"""carousel""",2,0.8,true,17,1,282,false,"""Image may contain: 2 people, indoor | Image may co…"
"""Bmd445DFh7t""","""00s_supermodels""","""fashion""",2031,53,2520,false,"""2018-08-14""",1534266258,64,0,"""carousel""",4,0.8,false,18,1,255,false,"""Image may contain: 1 person, closeup | Image may c…"


## 2. Qualidade e Integridade dos Dados

In [28]:
print("Valores Nulos por Coluna:")
df.null_count()

Valores Nulos por Coluna:


post_id,username,inf_category,followers,followees,posts_total,is_verified,data,timestamp_unix,likes,comments_count,post_type,n_imagens,aspect_ratio,is_sponsored,n_hashtags,n_usertags,caption_len,has_location,accessibility
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [29]:
posts_por_perfil = df.group_by("username").agg(pl.len().alias("n_posts")).sort("n_posts", descending=True)
posts_por_perfil.describe()

statistic,username,n_posts
str,str,f64
"""count""","""33840""",33840.0
"""null_count""","""0""",0.0
"""mean""",null,287.417199
"""std""",null,37.518166
"""min""","""00_rocketgirl""",1.0
"""25%""",null,297.0
"""50%""",null,300.0
"""75%""",null,300.0
"""max""","""zyndl_lala""",300.0


## 3. Cobertura do Lookup de Influenciadores
A coluna `perfil_completo_lookup` indica se o perfil do post possui informações completas (followers, category) no dataset de influencers. Substitui a antiga flag `perfilnolookup` para maior clareza semântica.

In [30]:
df_inf = pl.read_csv(CSV_INFLUENCERS)
perfis_completos = df_inf.filter(pl.col("Followers") > 0)["Username"].unique().to_list()

df = df.with_columns(
    pl.col("username").is_in(perfis_completos).alias("perfil_completo_lookup")
)
print("Posts com perfil completo no lookup:", df["perfil_completo_lookup"].sum())

Posts com perfil completo no lookup: 9566651


## 4. Cobertura Temporal

In [31]:
df = df.with_columns(
    pl.col("data").str.to_date("%Y-%m-%d", strict=False).alias("data_dt")
)
df.filter(pl.col("data_dt").is_not_null())['data_dt'].min(), df.filter(pl.col("data_dt").is_not_null())['data_dt'].max()

(datetime.date(2012, 2, 10), datetime.date(2019, 5, 15))

In [32]:
df = df.with_columns(pl.col("data_dt").dt.year().alias("ano"))
dist_anual = df.group_by("ano").agg(pl.len().alias("n_posts")).sort("ano")
dist_anual = dist_anual.with_columns((pl.col("n_posts") / pl.col("n_posts").sum() * 100).alias("pct"))
dist_anual

ano,n_posts,pct
i32,u32,f64
2012,1649,0.016954
2013,10421,0.107144
2014,35241,0.362331
2015,117036,1.203307
2016,514954,5.294505
2017,1805339,18.561611
2018,5249771,53.975572
2019,1991787,20.478578


**Justificativa Temporal**: O período de 2017 a 2019 concentra a grande maioria dos dados, compondo o triênio núcleo para as análises.

## 5. Diagnóstico de Zeros e Decisão Metodológica
A métrica de comments possui uma proporção massiva de zeros. O tratamento correto não é a exclusão da base, pois isso causa viés de seleção severo. Os zeros são métricas reais de baixo engajamento.

**Conclusão**: O filtro `comments_count > 0` não será aplicado à base principal, sendo restrito à camada de robustez `core_2017_2019`.

In [33]:
likes_zero = df.filter(pl.col("likes") == 0).shape[0]
comments_zero = df.filter(pl.col("comments_count") == 0).shape[0]
total = df.shape[0]

print(f"Posts com likes = 0 : {likes_zero} ({likes_zero/total*100:.2f}%)")
print(f"Posts com comments = 0 : {comments_zero} ({comments_zero/total*100:.2f}%)")

Posts com likes = 0 : 3044 (0.03%)
Posts com comments = 0 : 8830004 (90.79%)


## 6. Definição das Camadas do Dataset

In [34]:
base_completa = df.drop("ano")
print(f"Base Completa: {base_completa.shape}")

Base Completa: (9726198, 22)


In [35]:
base_2017_2019 = base_completa.filter(
    pl.col("data_dt").is_not_null(),
    pl.col("data_dt").dt.year().is_between(2017, 2019),
    pl.col("followers") > 0
)
print(f"Base 2017-2019 (Principal): {base_2017_2019.shape}")

Base 2017-2019 (Principal): (8897233, 22)


In [36]:
print("O dataset `core_2017_2019` será criado mais adiante, após a inclusão das métricas.")

O dataset `core_2017_2019` será criado mais adiante, após a inclusão das métricas.


## 7. Métricas Derivadas na Base Principal

In [37]:
base_2017_2019 = base_2017_2019.with_columns([
    ((pl.col("likes") + pl.col("comments_count")) / pl.col("followers") * 100).alias("er_classico"),
    ((pl.col("likes") + pl.col("comments_count") * 2) / pl.col("followers") * 100).alias("er_weighted")
])
base_2017_2019.select(["er_classico", "er_weighted"]).describe()

statistic,er_classico,er_weighted
str,f64,f64
"""count""",8.897233e6,8.897233e6
"""null_count""",0.0,0.0
"""mean""",4.327969,4.349675
"""std""",5.291196,5.33094
"""min""",0.0,0.0
"""25%""",1.569472,1.575025
"""50%""",3.020962,3.033254
"""75%""",5.491529,5.516313
"""max""",1408.70425,1408.70425


## 8. Segmentação e Buckets

In [38]:
base_2017_2019 = base_2017_2019.with_columns(
    pl.when(pl.col("followers") <= 10_000).then(pl.lit("nano"))
    .when(pl.col("followers") <= 100_000).then(pl.lit("micro"))
    .when(pl.col("followers") <= 500_000).then(pl.lit("mid-tier"))
    .when(pl.col("followers") <= 1_000_000).then(pl.lit("macro"))
    .otherwise(pl.lit("mega")).alias("bucket_followers")
)

In [39]:
base_2017_2019 = base_2017_2019.with_columns(
    pl.when(pl.col("er_classico") < 1).then(pl.lit("LOW"))
    .when(pl.col("er_classico") < 3.5).then(pl.lit("MEDIUM"))
    .when(pl.col("er_classico") < 6).then(pl.lit("HIGH"))
    .otherwise(pl.lit("VIRAL")).alias("er_bucket_industria")
)

In [40]:
q25 = base_2017_2019["er_classico"].quantile(0.25)
q50 = base_2017_2019["er_classico"].quantile(0.50)
q75 = base_2017_2019["er_classico"].quantile(0.75)

base_2017_2019 = base_2017_2019.with_columns(
    pl.when(pl.col("er_classico") <= q25).then(pl.lit("Q1"))
    .when(pl.col("er_classico") <= q50).then(pl.lit("Q2"))
    .when(pl.col("er_classico") <= q75).then(pl.lit("Q3"))
    .otherwise(pl.lit("Q4")).alias("er_bucket_percentil")
)

In [41]:
thresholds = (
    base_2017_2019.filter(pl.col("er_classico") > 0)
    .group_by("bucket_followers")
    .agg([
        pl.col("er_classico").mean().alias("er_media_bucket"),
        pl.col("er_classico").std().alias("er_std_bucket"),
    ])
    .with_columns([
        (pl.col("er_media_bucket") - pl.col("er_std_bucket")).alias("thresh_low"),
        (pl.col("er_media_bucket") + pl.col("er_std_bucket")).alias("thresh_high"),
    ])
)

base_2017_2019 = base_2017_2019.join(thresholds, on="bucket_followers", how="left")

base_2017_2019 = base_2017_2019.with_columns(
    pl.when(pl.col("er_classico") < pl.col("thresh_low")).then(pl.lit("LOW"))
    .when(pl.col("er_classico") < pl.col("thresh_high")).then(pl.lit("MEDIUM"))
    .when(pl.col("er_classico") < (pl.col("thresh_high") + pl.col("er_std_bucket"))).then(pl.lit("HIGH"))
    .otherwise(pl.lit("VIRAL")).alias("adapted_bucket")
)

## 9. Métricas de Estabilidade (CV) no Core
Calculadas apenas sobre a camada `core_2017_2019` para maior robustez.

In [42]:
core_2017_2019 = base_2017_2019.filter(pl.col("comments_count") > 0)
print(f"Core 2017-2019 (Robustez): {core_2017_2019.shape}")

Core 2017-2019 (Robustez): (825268, 32)


In [43]:
cv_core = (
    core_2017_2019.group_by("username")
    .agg([
        pl.len().alias("n_posts_validos"),
        pl.col("er_classico").mean().alias("er_media"),
        pl.col("er_classico").std().alias("er_std"),
        pl.col("er_classico").median().alias("er_mediana"),
    ])
    .filter(pl.col("n_posts_validos") >= 5)
    .with_columns(
        pl.when(pl.col("er_media") == 0).then(None)
        .otherwise(pl.col("er_std") / pl.col("er_media")).alias("cv")
    )
)
cv_core.head()

username,n_posts_validos,er_media,er_std,er_mediana,cv
str,u32,f64,f64,f64,f64
"""thewhitmore""",30,0.585827,0.509956,0.437345,0.87049
"""starpicsofficial""",16,0.343555,0.140277,0.330639,0.408309
"""helenanderz""",24,4.181193,3.016139,3.512886,0.721358
"""tribe_stewart""",24,3.794501,1.325207,3.693146,0.349244
"""tazerleejones""",7,9.06114,0.392079,9.050577,0.04327


In [44]:
cv_anual = (
    core_2017_2019.with_columns(pl.col("data_dt").dt.year().alias("ano"))
    .group_by(["username", "ano"])
    .agg([
        pl.len().alias("n_posts_validos"),
        pl.col("er_classico").mean().alias("er_media"),
        pl.col("er_classico").std().alias("er_std"),
    ])
    .filter(pl.col("n_posts_validos") >= 5)
    .with_columns(
        pl.when(pl.col("er_media") == 0).then(None)
        .otherwise(pl.col("er_std") / pl.col("er_media")).alias("cv")
    )
)

## 10. Persistência

In [45]:
base_completa.write_parquet(PARQUET_BASE_COMPLETA, compression="zstd")
base_2017_2019.write_parquet(PARQUET_BASE_2017_2019, compression="zstd")
core_2017_2019.write_parquet(PARQUET_CORE_2017_2019, compression="zstd")
cv_core.write_parquet(PARQUET_CV_CORE, compression="zstd")
cv_anual.write_parquet(PARQUET_CV_ANUAL, compression="zstd")
print("Bases persistidas com sucesso em parquets zstd!")

Bases persistidas com sucesso em parquets zstd!
